In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import regionmask as rm
from pathlib import Path

import sys, os
module_path = "../bin"
sys.path.insert(0, module_path)

import um_utils_plotting as umplt
import um_utils as um
from geometry_fix import apply_geometry_collection_fix

apply_geometry_collection_fix()

um.WarningSuppress()

MEANS_DIR = Path("../Data/Models/Monthly_Means")
PROXY_DIR = Path("/Users/ajw1g19/Library/CloudStorage/OneDrive-UniversityofSouthampton/INSPIRE DTP/Projects/Palaeo-Proxies/")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## RMSE Scores for Precipitation

In [ ]:
#------------------------------------------------------------
# Load Bartlein and LegacyClimate precipitation
# reconstructions
#------------------------------------------------------------

BART_MAP = um.importUMData(PROXY_DIR / "Bartlein" / "map_delta_06ka_ALL_grid_2x2_ex.nc")

BART = BART_MAP["map_anm_mean"].stack(points=("lat", "lon")).dropna("points")

LC_anoms_df = pd.read_excel(f"../Data/Proxies/LC_climate_reconstruction_anomalies.xlsx")
LC_anoms_df = LC_anoms_df[LC_anoms_df["Continent"].isin(["Western North America", "Eastern North America"])]

LC_lat = LC_anoms_df["Latitude"].values
LC_lon = LC_anoms_df["Longitude"].values
LC = LC_anoms_df["Precip annual total [mm/a] (MAT)"]

# Combine proxy datasets
FULL = np.concatenate([BART.values,LC])
FULL_LON = np.concatenate([BART["lon"].values, LC_lon])
FULL_LAT = np.concatenate([BART["lat"].values, LC_lat])

# Prep dicts to store RMSE values
pmip_means_dir = "/Users/ajw1g19/Library/CloudStorage/OneDrive-UniversityofSouthampton/INSPIRE DTP/Projects/CMIP_Data/PMIP4_Monthly_Means"
models = [fpath.split(".")[0] for fpath in sorted(os.listdir(pmip_means_dir))]
models = ["UKESM1-1-LL", "PMIP4_Ensemble"] + models[:-1]

categories = ["GP", "N (GP)", "NAM", "N (NAM)", "WNA", "N (WNA)", "CNA", "N (CNA)", "ENA", "N (ENA)", "NWN", "N (NWN)", "NEN", "N (NEN)"]

bart_dict = {cat: np.zeros(len(models)) for cat in categories}
lc_dict = {cat: np.zeros(len(models)) for cat in categories}
full_dict = {cat: np.zeros(len(models)) for cat in categories}

#------------------------------------------------------------
# Iterate through the models and interpolate precipitation
# anomalies to the proxy points for each subregion
# Then calculate the RMSE between the two
#------------------------------------------------------------

for i, model in enumerate(models):
    if model == "UKESM1-1-LL":
        mh_pr = um.importUMData(MEANS_DIR / "MH_monthly_means.nc")["pr"]
        pi_pr = um.importUMData(MEANS_DIR / "PI_monthly_means.nc")["pr"]

        model_anom = (mh_pr.mean(dim="time") - pi_pr.mean(dim="time")) * 12
        lsm = um.importUMData(f"../Data/lsm_um13.2.nc")

    else:
        modelMeans = um.importUMData(f"{pmip_means_dir}/{model}.nc")

        model_anom = (modelMeans["mh_pr"].mean(dim="time") - modelMeans["pi_pr"].mean(dim="time")) * 12
        lsm = modelMeans["lsm"]

    for j, abbrev in enumerate(["NWN", "NEN", "WNA", "CNA", "ENA", "NAM", "GP"]):
        if abbrev in ["GP", "NAM"]: region = umplt.umRegion(abbrev)
        else: region = rm.defined_regions.ar6.all[[abbrev]]

        BART_RG, BART_RG_LON, BART_RG_LAT = umplt.proxyRegionMask(BART.values, region, lon_arr=BART["lon"].values, lat_arr=BART["lat"].values, landsea_mask=lsm)
        LC_RG, LC_RG_LON, LC_RG_LAT = umplt.proxyRegionMask(LC, region, lon_arr=LC_lon, lat_arr=LC_lat, landsea_mask=lsm)
        FULL_RG, FULL_RG_LON, FULL_RG_LAT = umplt.proxyRegionMask(FULL, region, lon_arr=FULL_LON, lat_arr=FULL_LAT, landsea_mask=lsm)

        model_to_bart = model_anom.interp(latitude=("points", BART_RG_LAT), longitude=("points", BART_RG_LON), method="linear")
        model_to_lc = model_anom.interp(latitude=("points", LC_RG_LAT), longitude=("points", LC_RG_LON), method="linear") 
        model_to_full = model_anom.interp(latitude=("points", FULL_RG_LAT), longitude=("points", FULL_RG_LON), method="linear")

        bart_rmse = np.sqrt(np.nanmean((model_to_bart - BART_RG)**2))  
        lc_rmse = np.sqrt(np.nanmean((model_to_lc - LC_RG)**2)) 
        full_rmse = np.sqrt(np.nanmean((model_to_full - FULL_RG)**2)) 

        bart_dict[abbrev][i] = bart_rmse
        bart_dict[f"N ({abbrev})"][i] = int(len(BART_RG))
        
        lc_dict[abbrev][i] = lc_rmse
        lc_dict[f"N ({abbrev})"][i] = int(len(LC_RG))

        full_dict[abbrev][i] = full_rmse
        full_dict[f"N ({abbrev})"][i] = int(len(FULL_RG))

bart_df = pd.DataFrame(bart_dict, index=models)
bart_df.index.name = "Models"

lc_df = pd.DataFrame(lc_dict, index=models)
lc_df.index.name = "Models"

full_df = pd.DataFrame(full_dict, index=models)
full_df.index.name = "Models"

out_file = "../Data/Proxies/pr_rmse_vals.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    bart_df.to_excel(writer, sheet_name="Bartlein", index=True)
    lc_df.to_excel(writer, sheet_name="LegacyClimate", index=True)
    full_df.to_excel(writer, sheet_name="Combined", index=True)


## RMSE Scores for Air Temperature

In [ ]:
#------------------------------------------------------------
# Load Bartlein and LegacyClimate tas reconstructions
#------------------------------------------------------------

BART_MAT = um.importUMData(PROXY_DIR / "Bartlein" / "mat_delta_06ka_ALL_grid_2x2_ex.nc")

BART = BART_MAT["mat_anm_mean"].stack(points=("lat", "lon")).dropna("points")

LC_anoms_df = pd.read_excel(f"../Data/Proxies/LC_climate_reconstruction_anomalies.xlsx")
LC_anoms_df = LC_anoms_df[LC_anoms_df["Continent"].isin(["Western North America", "Eastern North America"])]

LC_lat = LC_anoms_df["Latitude"].values
LC_lon = LC_anoms_df["Longitude"].values
LC = LC_anoms_df["T air (annual mean) [°C] (MAT)"]

# Combine proxy datasets
FULL = np.concatenate([BART.values, LC])
FULL_LON = np.concatenate([BART["lon"].values, LC_lon])
FULL_LAT = np.concatenate([BART["lat"].values, LC_lat])

# Prep dicts to store RMSE values
pmip_means_dir = "/Users/ajw1g19/Library/CloudStorage/OneDrive-UniversityofSouthampton/INSPIRE DTP/Projects/CMIP_Data/PMIP4_Monthly_Means"
models = [fpath.split(".")[0] for fpath in sorted(os.listdir(pmip_means_dir))]
models = ["UKESM1-1-LL", "PMIP4_Ensemble"] + models[:-1]

categories = ["GP", "N (GP)", "NAM", "N (NAM)", "WNA", "N (WNA)", "CNA", "N (CNA)", "ENA", "N (ENA)", "NWN", "N (NWN)", "NEN", "N (NEN)"]

bart_dict = {cat: np.zeros(len(models)) for cat in categories}
lc_dict = {cat: np.zeros(len(models)) for cat in categories}
full_dict = {cat: np.zeros(len(models)) for cat in categories}

#------------------------------------------------------------
# Iterate through the models and interpolate tas anomalies
# to the proxy points for each subregion
# Then calculate the RMSE between the two
#------------------------------------------------------------

for i, model in enumerate(models):
    if model == "UKESM1-1-LL":
        mh_tas = um.importUMData(MEANS_DIR / "MH_monthly_means.nc")["tas"]
        pi_tas = um.importUMData(MEANS_DIR / "PI_monthly_means.nc")["tas"]

        model_anom = mh_tas.mean(dim="time") - pi_tas.mean(dim="time")
        lsm = um.importUMData(f"../Data/lsm_um13.2.nc")

    else:
        modelMeans = um.importUMData(f"{pmip_means_dir}/{model}.nc")

        model_anom = (modelMeans["mh_tas"].mean(dim="time") - modelMeans["pi_tas"].mean(dim="time"))
        lsm = modelMeans["lsm"]

    for j, abbrev in enumerate(["NWN", "NEN", "WNA", "CNA", "ENA", "NAM", "GP"]):
        if abbrev in ["GP", "NAM"]: region = umplt.umRegion(abbrev)
        else: region = rm.defined_regions.ar6.all[[abbrev]]

        BART_RG, BART_RG_LON, BART_RG_LAT = umplt.proxyRegionMask(BART.values, region, lon_arr=BART["lon"].values, lat_arr=BART["lat"].values, landsea_mask=lsm)
        LC_RG, LC_RG_LON, LC_RG_LAT = umplt.proxyRegionMask(LC, region, lon_arr=LC_lon, lat_arr=LC_lat, landsea_mask=lsm)
        FULL_RG, FULL_RG_LON, FULL_RG_LAT = umplt.proxyRegionMask(FULL, region, lon_arr=FULL_LON, lat_arr=FULL_LAT, landsea_mask=lsm)

        model_to_bart = model_anom.interp(latitude=("points", BART_RG_LAT), longitude=("points", BART_RG_LON), method="linear")
        model_to_lc = model_anom.interp(latitude=("points", LC_RG_LAT), longitude=("points", LC_RG_LON), method="linear") 
        model_to_full = model_anom.interp(latitude=("points", FULL_RG_LAT), longitude=("points", FULL_RG_LON), method="linear")

        bart_rmse = np.sqrt(np.nanmean((model_to_bart - BART_RG)**2))  
        lc_rmse = np.sqrt(np.nanmean((model_to_lc - LC_RG)**2)) 
        full_rmse = np.sqrt(np.nanmean((model_to_full - FULL_RG)**2)) 

        bart_dict[abbrev][i] = bart_rmse
        bart_dict[f"N ({abbrev})"][i] = int(len(BART_RG))
        
        lc_dict[abbrev][i] = lc_rmse
        lc_dict[f"N ({abbrev})"][i] = int(len(LC_RG))

        full_dict[abbrev][i] = full_rmse
        full_dict[f"N ({abbrev})"][i] = int(len(FULL_RG))

bart_df = pd.DataFrame(bart_dict, index=models)
bart_df.index.name = "Models"

lc_df = pd.DataFrame(lc_dict, index=models)
lc_df.index.name = "Models"

full_df = pd.DataFrame(full_dict, index=models)
full_df.index.name = "Models"

out_file = "../Data/Proxies/tas_rmse_vals.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    bart_df.to_excel(writer, sheet_name="Bartlein", index=True)
    lc_df.to_excel(writer, sheet_name="LegacyClimate", index=True)
    full_df.to_excel(writer, sheet_name="Combined", index=True)


In [ ]:
#------------------------------------------------------------
# RMSE scores for vegetation changes
#------------------------------------------------------------

#------------------------------------------------------------
# Load UKESM and Dawson (pollen) vegetation fractions
#------------------------------------------------------------

mh_sfrac = um.importUMData(MEANS_DIR / "MH_monthly_means.nc")["surfacefrac"] * 100
pi_sfrac = um.importUMData(MEANS_DIR / "PI_monthly_means.nc")["surfacefrac"] * 100
lsm = um.importUMData(f"../Data/lsm_um13.2.nc")

# Dawson Vegetation Reconstruction
dawson_vars = ["ET", "ST", "OL"]
df200, df6k = um.importRevealsYrs(PROXY_DIR / "Dawson" / "REVEALS_LCT_gridded.csv", [200, 6000])
dawson_lon, dawson_lat = df200["x"].values, df200["y"].values
for var in dawson_vars:
    df200[var] = df200[var] * 100
    df6k[var] = df6k[var] * 100

PFTS = {"Evergreen Forest": [1, 2, 4, 12], "Summergreen Forest":[0, 3, 11], "Open Land": [5, 6, 7, 8, 9, 10]}
categories = ["GP", "N (GP)", "NAM", "N (NAM)", "WNA", "N (WNA)", "CNA", "N (CNA)", "ENA", "N (ENA)", "NWN", "N (NWN)", "NEN", "N (NEN)"]

rmse_dict = {cat: np.zeros(len(PFTS)) for cat in categories}

#------------------------------------------------------------
# Iterate through the three land cover types and interpolate
# UKESM surface fractions to Dawson sample locations
# Calculate RMSE for each subregion
#------------------------------------------------------------

for i, pft in enumerate(PFTS.keys()):
    model_anom = mh_sfrac[:, PFTS[pft]].sum(dim="surface_tiles", min_count=1).mean(dim="time") - pi_sfrac[:, PFTS[pft]].sum(dim="surface_tiles", min_count=1).mean(dim="time")

    DAWSON = df6k[dawson_vars[i]].values - df200[dawson_vars[i]].values

    for j, abbrev in enumerate(["NWN", "NEN", "WNA", "CNA", "ENA", "NAM", "GP"]):
        if abbrev in ["GP", "NAM"]: region = umplt.umRegion(abbrev)
        else: region = rm.defined_regions.ar6.all[[abbrev]]

        DAWSON_RG, DAWSON_RG_LON, DAWSON_RG_LAT = umplt.proxyRegionMask(DAWSON, region, lon_arr=dawson_lon, lat_arr=dawson_lat, landsea_mask=lsm)

        model_to_dawson = model_anom.interp(latitude=("points", DAWSON_RG_LAT), longitude=("points", DAWSON_RG_LON), method="linear")

        veg_rmse = np.sqrt(np.nanmean((model_to_dawson - DAWSON_RG)**2))

        rmse_dict[abbrev][i] = veg_rmse
        rmse_dict[f"N ({abbrev})"][i] = len(DAWSON_RG)

rmse_df = pd.DataFrame(rmse_dict, index=["Evergreen Forest", "Summergreen Forest", "Open Land"])
rmse_df.index.name = "PFT Type"
rmse_df.to_excel("../Data/Proxies/veg_rmse_vals.xlsx", index=True)